# Balance-Guided Oblique Trees: End-to-End Benchmark Demo

This notebook demonstrates the **Balance-Guided Oblique Trees** experiment, which compares three tree-based methods on tabular data:

1. **Axis-Aligned FIGS** - Standard interpretable tree ensembles (baseline)
2. **Random Oblique FIGS** - Oblique splits with random feature subsets (baseline)
3. **Signed-Spectral FIGS** - Oblique splits guided by SPONGE spectral clustering of the Co-Information matrix (our method)

The key idea: we compute a **Co-Information (CoI) matrix** between features and the target, then use **SPONGE signed spectral clustering** to discover feature modules (groups of synergistic/redundant features). These modules guide which features are combined in oblique splits.

**Pipeline**: Data → CoI Matrix → SPONGE Clustering → Feature Modules → Oblique FIGS Training → Evaluation

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.14.1', 'matplotlib==3.10.0', 'tabulate==0.9.0', 'joblib==1.4.2')

In [ ]:
import gc
import json
import math
import os
import sys
import time
import warnings
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy.linalg import eigh, eigvalsh
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.linear_model import Ridge
from sklearn.metrics import balanced_accuracy_score, r2_score, roc_auc_score, silhouette_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")

## Data Loading

Load the demo dataset (electricity — 7 features, binary classification). The data is fetched from GitHub with a local fallback.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7dffcf-balance-guided-oblique-trees-signed-spec/main/experiment_iter2_end_to_end_bala/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded {len(data['datasets'])} dataset(s)")
for ds in data['datasets']:
    print(f"  {ds['dataset']}: {len(ds['examples'])} examples")

## Configuration

All tunable parameters are defined here. Values are set to minimal defaults for fast demo execution. Original full-experiment values are shown in comments.

In [ ]:
# --- Demo config (minimum values for fast execution) ---
N_FOLDS = 2                  # Original: 5
MAX_SPLITS_GRID = [5]        # Original: [5, 10, 15, 20]
COI_K_NEIGHBORS = 3          # Original: 5
COI_SUBSAMPLE_N = 80         # Original: 10000
SPONGE_K_MAX = 4             # Original: 10
SPONGE_TAU_P = 1.0
SPONGE_TAU_N = 1.0
SPONGE_EPS = 1e-10
NUM_REPETITIONS = 2          # Original: 5
BEAM_SIZE_DEFAULT = 3        # Original: 5
MIN_SAMPLES_LEAF = 2         # Original: 5
RIDGE_ALPHA = 1.0
RANDOM_STATE = 42
N_JOBS = 1                   # Original: max(1, NUM_CPUS - 1)
METHODS = ["axis_aligned_figs", "random_oblique_figs", "signed_spectral_figs"]

## Data Parsing

Parse the loaded JSON data into NumPy arrays (X, y, folds) for training/testing.

In [ ]:
def parse_datasets(raw_data):
    """Parse JSON data into NumPy arrays, matching original load_all_datasets()."""
    datasets = {}
    for ds_block in raw_data["datasets"]:
        ds_name = ds_block["dataset"]
        examples = ds_block["examples"]
        feature_names = list(json.loads(examples[0]["input"]).keys())
        task_type = examples[0]["metadata_task_type"]

        X_rows, y_vals, fold_ids = [], [], []
        for ex in examples:
            feat_dict = json.loads(ex["input"])
            X_rows.append([feat_dict[fn] for fn in feature_names])
            y_vals.append(ex["output"])
            fold_ids.append(ex["metadata_fold"])

        X = np.array(X_rows, dtype=np.float64)
        folds = np.array(fold_ids, dtype=int)

        if task_type == "classification":
            y = np.array([int(v) for v in y_vals], dtype=int)
            n_classes = examples[0].get("metadata_n_classes", len(set(y_vals)))
        else:
            y = np.array([float(v) for v in y_vals], dtype=np.float64)
            n_classes = None

        # Replace NaN with 0
        nan_count = np.isnan(X).sum()
        if nan_count > 0:
            X = np.nan_to_num(X, nan=0.0)

        datasets[ds_name] = {
            "X": X, "y": y, "folds": folds,
            "task_type": task_type,
            "feature_names": feature_names,
            "n_classes": n_classes,
        }
        print(f"  {ds_name}: n={len(y)}, d={X.shape[1]}, task={task_type}, "
              f"classes={n_classes}, folds={sorted(np.unique(folds).tolist())}")
    return datasets

datasets = parse_datasets(data)

## Co-Information Matrix Computation

Compute the Co-Information matrix: `CoI(Xi, Xj; Y) = I(Xi;Y) + I(Xj;Y) - I(Xi,Xj;Y)`.
Positive entries indicate **redundancy**, negative entries indicate **synergy** between feature pairs.

In [ ]:
def _mi_individual(Xi, y, task_type, k=5):
    """MI between single feature Xi and target y using sklearn."""
    if Xi.ndim == 1:
        Xi = Xi.reshape(-1, 1)
    if task_type == "classification":
        from sklearn.feature_selection import mutual_info_classif
        val = mutual_info_classif(Xi, y, n_neighbors=k, random_state=RANDOM_STATE)[0]
    else:
        from sklearn.feature_selection import mutual_info_regression
        val = mutual_info_regression(Xi, y, n_neighbors=k, random_state=RANDOM_STATE)[0]
    return max(0.0, float(val))


def _mi_joint(Xi, Xj, y, task_type, k=5):
    """Compute I({Xi,Xj}; Y) for one feature pair."""
    X_joint = np.column_stack([Xi, Xj])
    if task_type == "classification":
        from sklearn.feature_selection import mutual_info_classif
        val = mutual_info_classif(X_joint[:, :1], y, n_neighbors=k, random_state=RANDOM_STATE)[0]
        # Approximate joint MI using PCA projection
        from sklearn.decomposition import PCA
        proj = PCA(n_components=1).fit_transform(X_joint)
        return max(0.0, float(mutual_info_classif(proj, y, n_neighbors=k, random_state=RANDOM_STATE)[0]))
    else:
        from sklearn.feature_selection import mutual_info_regression
        from sklearn.decomposition import PCA
        proj = PCA(n_components=1).fit_transform(X_joint)
        return float(mutual_info_regression(proj, y, n_neighbors=k, random_state=RANDOM_STATE)[0])


def compute_coi_matrix(X, y, task_type, k=5, n_jobs=1):
    """Compute d x d Co-Information matrix."""
    n, d = X.shape

    # Step 1: Individual MI
    individual_mi = np.zeros(d)
    for i in range(d):
        individual_mi[i] = _mi_individual(X[:, i], y, task_type, k)

    # Step 2: All-pairs joint MI
    pairs = [(i, j) for i in range(d) for j in range(i + 1, d)]
    joint_mi_values = [_mi_joint(X[:, i], X[:, j], y, task_type, k) for i, j in pairs]

    # Step 3: Assemble CoI matrix
    coi_matrix = np.zeros((d, d))
    for idx, (i, j) in enumerate(pairs):
        coi = individual_mi[i] + individual_mi[j] - joint_mi_values[idx]
        coi_matrix[i, j] = coi
        coi_matrix[j, i] = coi

    return coi_matrix, individual_mi

print("Co-Information functions defined.")

## SPONGE Signed Spectral Clustering

Use the SPONGE_sym algorithm to cluster features based on the signed Co-Information graph. Positive edges = redundancy, negative edges = synergy. Selects optimal k via eigengap + silhouette.

In [ ]:
def _safe_sqrt_inv_diag(D, eps=1e-10):
    dv = np.diag(D)
    inv_s = np.zeros_like(dv)
    nz = dv > eps
    inv_s[nz] = 1.0 / np.sqrt(dv[nz])
    return np.diag(inv_s)


def sponge_sym(coi_matrix, k, tau_p=1.0, tau_n=1.0, eps=1e-10):
    """SPONGE_sym via generalized eigenvalue problem."""
    d = coi_matrix.shape[0]
    A_pos = np.maximum(coi_matrix, 0)
    A_neg = np.maximum(-coi_matrix, 0)
    D_pos = np.diag(A_pos.sum(axis=1))
    D_neg = np.diag(A_neg.sum(axis=1))
    L_pos = D_pos - A_pos
    L_neg = D_neg - A_neg

    Dp_isq = _safe_sqrt_inv_diag(D_pos, eps)
    Dn_isq = _safe_sqrt_inv_diag(D_neg, eps)
    L_sym_pos = Dp_isq @ L_pos @ Dp_isq
    L_sym_neg = Dn_isq @ L_neg @ Dn_isq

    A_mat = L_sym_pos + tau_n * np.eye(d)
    B_mat = L_sym_neg + tau_p * np.eye(d) + eps * np.eye(d)

    eigenvalues, eigenvectors = eigh(A_mat, b=B_mat, subset_by_index=[0, k - 1])

    V = eigenvectors
    norms = np.linalg.norm(V, axis=1, keepdims=True)
    norms[norms < eps] = 1.0
    V_norm = V / norms

    labels = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE).fit_predict(V_norm)
    return labels, eigenvalues


def select_k_and_cluster(coi_matrix, k_max=10):
    """Select optimal k via eigengap + silhouette, then run SPONGE_sym."""
    d = coi_matrix.shape[0]
    k_max = min(k_max, max(2, d // 3), d - 1)
    if k_max < 2:
        k_max = 2

    A_pos = np.maximum(coi_matrix, 0)
    A_neg = np.maximum(-coi_matrix, 0)
    D_pos = np.diag(A_pos.sum(axis=1))
    D_neg = np.diag(A_neg.sum(axis=1))
    L_pos = D_pos - A_pos
    L_neg = D_neg - A_neg

    eps = SPONGE_EPS
    Dp_isq = _safe_sqrt_inv_diag(D_pos, eps)
    Dn_isq = _safe_sqrt_inv_diag(D_neg, eps)
    L_sym_pos = Dp_isq @ L_pos @ Dp_isq
    L_sym_neg = Dn_isq @ L_neg @ Dn_isq

    A_mat = L_sym_pos + SPONGE_TAU_N * np.eye(d)
    B_mat = L_sym_neg + SPONGE_TAU_P * np.eye(d) + eps * np.eye(d)

    all_evals, all_evecs = eigh(A_mat, b=B_mat, subset_by_index=[0, k_max - 1])

    gaps = np.diff(all_evals)
    if len(gaps) == 0:
        labels, evals = sponge_sym(coi_matrix, k=2)
        return labels, evals, 2, -1.0

    top3 = np.argsort(gaps)[-3:]
    candidates = sorted(set(idx + 1 for idx in top3 if idx + 1 >= 2))
    if not candidates:
        candidates = [2]

    best_k, best_sil = candidates[0], -1.0
    for kc in candidates:
        V = all_evecs[:, :kc]
        norms = np.linalg.norm(V, axis=1, keepdims=True)
        norms[norms < 1e-10] = 1.0
        V_n = V / norms
        labs = KMeans(n_clusters=kc, n_init=20, random_state=RANDOM_STATE).fit_predict(V_n)
        if len(set(labs)) >= 2:
            sil = silhouette_score(V_n, labs)
            if sil > best_sil:
                best_sil = sil
                best_k = kc

    if best_sil < 0.0:
        best_k = max(2, int(np.ceil(np.sqrt(d / 2))))
        best_k = min(best_k, k_max)

    labels, eigenvalues = sponge_sym(coi_matrix, k=best_k)
    return labels, eigenvalues, best_k, best_sil


def compute_frustration_index(coi_matrix):
    """Frustration = lambda_min / lambda_max of signed Laplacian."""
    D_bar = np.diag(np.sum(np.abs(coi_matrix), axis=1))
    L_sigma = D_bar - coi_matrix
    evals = eigvalsh(L_sigma)
    lam_min, lam_max = evals[0], evals[-1]
    if lam_max < 1e-12:
        return 0.0
    return float(max(0.0, lam_min) / lam_max)


def extract_modules(labels, d):
    modules = {}
    for i in range(d):
        modules.setdefault(int(labels[i]), []).append(i)
    return list(modules.values())

print("SPONGE clustering functions defined.")

## Oblique FIGS Tree Framework

The core tree-building algorithm: a greedy tree-sum ensemble that supports oblique (multi-feature) splits. Three variants:
- **AxisAlignedFIGSCustom**: Only axis-aligned splits (baseline)
- **RandomObliqueFIGS**: Oblique splits from random feature subsets
- **SignedSpectralFIGS**: Oblique splits guided by spectral modules (our method)

In [ ]:
@dataclass
class ObliqueFIGSNode:
    feature: int = -1
    features: list = field(default_factory=list)
    weights: np.ndarray = field(default_factory=lambda: np.array([]))
    threshold: float = 0.0
    is_oblique: bool = False
    value: float = 0.0
    left: Optional["ObliqueFIGSNode"] = None
    right: Optional["ObliqueFIGSNode"] = None
    n_samples: int = 0

    @property
    def is_leaf(self):
        return self.left is None and self.right is None


def _predict_single(node, x):
    while not node.is_leaf:
        if node.is_oblique:
            proj = np.dot(x[node.features], node.weights)
        else:
            proj = x[node.feature]
        node = node.left if proj <= node.threshold else node.right
    return node.value


def _predict_tree(node, X):
    return np.array([_predict_single(node, X[i]) for i in range(len(X))])


def _get_leaves_and_masks(node, X):
    leaves, masks = [], []
    def _recurse(nd, mask):
        if nd.is_leaf:
            leaves.append(nd); masks.append(mask); return
        if nd.is_oblique:
            proj = X[:, nd.features] @ nd.weights
        else:
            proj = X[:, nd.feature]
        _recurse(nd.left, mask & (proj <= nd.threshold))
        _recurse(nd.right, mask & (proj > nd.threshold))
    _recurse(node, np.ones(len(X), dtype=bool))
    return leaves, masks


def _fit_axis_aligned(X_leaf, res):
    if len(res) < 2 * MIN_SAMPLES_LEAF:
        return None
    stump = DecisionTreeRegressor(max_depth=1, min_samples_leaf=MIN_SAMPLES_LEAF, random_state=RANDOM_STATE)
    stump.fit(X_leaf, res)
    t = stump.tree_
    if t.feature[0] < 0 or t.n_leaves < 2:
        return None
    feat, thr = int(t.feature[0]), float(t.threshold[0])
    left_m = X_leaf[:, feat] <= thr
    n_l, n_r = int(left_m.sum()), int((~left_m).sum())
    if n_l < MIN_SAMPLES_LEAF or n_r < MIN_SAMPLES_LEAF:
        return None
    lv = float(np.mean(res[left_m]))
    rv = float(np.mean(res[~left_m]))
    gain = float(np.sum(res ** 2) - np.sum((res[left_m] - lv) ** 2) - np.sum((res[~left_m] - rv) ** 2))
    return {"gain": gain, "feature": feat, "threshold": thr, "is_oblique": False,
            "left_value": lv, "right_value": rv, "n_left": n_l, "n_right": n_r}


def _fit_oblique(X_leaf, res, feat_idx):
    if len(res) < 2 * MIN_SAMPLES_LEAF or len(feat_idx) < 2:
        return None
    X_sub = X_leaf[:, feat_idx]
    if np.var(X_sub, axis=0).max() < 1e-12:
        return None
    try:
        ridge = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True)
        ridge.fit(X_sub, res)
        proj = X_sub @ ridge.coef_
    except Exception:
        return None
    if np.var(proj) < 1e-12:
        return None
    stump = DecisionTreeRegressor(max_depth=1, min_samples_leaf=MIN_SAMPLES_LEAF, random_state=RANDOM_STATE)
    stump.fit(proj.reshape(-1, 1), res)
    t = stump.tree_
    if t.feature[0] < 0 or t.n_leaves < 2:
        return None
    thr = float(t.threshold[0])
    left_m = proj <= thr
    n_l, n_r = int(left_m.sum()), int((~left_m).sum())
    if n_l < MIN_SAMPLES_LEAF or n_r < MIN_SAMPLES_LEAF:
        return None
    lv = float(np.mean(res[left_m]))
    rv = float(np.mean(res[~left_m]))
    gain = float(np.sum(res ** 2) - np.sum((res[left_m] - lv) ** 2) - np.sum((res[~left_m] - rv) ** 2))
    return {"gain": gain, "features": list(feat_idx), "weights": ridge.coef_.copy(),
            "threshold": thr, "is_oblique": True,
            "left_value": lv, "right_value": rv, "n_left": n_l, "n_right": n_r}


class BaseFIGSOblique:
    """Greedy tree-sum ensemble with oblique split support."""
    def __init__(self, max_splits=10, beam_size=BEAM_SIZE_DEFAULT,
                 num_repetitions=NUM_REPETITIONS, random_state=RANDOM_STATE):
        self.max_splits = max_splits
        self.beam_size = beam_size
        self.num_repetitions = num_repetitions
        self.random_state = random_state
        self.trees_ = []
        self.complexity_ = 0

    def _get_feature_subsets(self, d, rng):
        raise NotImplementedError

    def fit(self, X, y):
        n, d = X.shape
        rng = np.random.default_rng(self.random_state)
        self.trees_ = []
        self.complexity_ = 0
        root = ObliqueFIGSNode(value=float(np.mean(y)), n_samples=n)
        self.trees_ = [root]

        for _ in range(self.max_splits):
            preds = np.zeros(n)
            for tree in self.trees_:
                preds += _predict_tree(tree, X)
            residuals = y - preds
            best_gain, best_action = 1e-10, None

            for tree in self.trees_:
                leaves, masks = _get_leaves_and_masks(tree, X)
                for leaf, mask in zip(leaves, masks):
                    idxs = np.where(mask)[0]
                    if len(idxs) < 2 * MIN_SAMPLES_LEAF:
                        continue
                    Xl, rl = X[idxs], residuals[idxs]
                    aa = _fit_axis_aligned(Xl, rl)
                    if aa and aa["gain"] > best_gain:
                        best_gain = aa["gain"]; best_action = ("split", leaf, aa, idxs)
                    for subset in self._get_feature_subsets(d, rng):
                        ob = _fit_oblique(Xl, rl, subset)
                        if ob and ob["gain"] > best_gain:
                            best_gain = ob["gain"]; best_action = ("split", leaf, ob, idxs)

            aa = _fit_axis_aligned(X, residuals)
            if aa and aa["gain"] > best_gain:
                best_gain = aa["gain"]; best_action = ("new", None, aa, np.arange(n))
            for subset in self._get_feature_subsets(d, rng):
                ob = _fit_oblique(X, residuals, subset)
                if ob and ob["gain"] > best_gain:
                    best_gain = ob["gain"]; best_action = ("new", None, ob, np.arange(n))

            if best_action is None:
                break
            act, leaf, info, _ = best_action
            if act == "new":
                self.trees_.append(self._make_node(info))
            else:
                self._split_leaf(leaf, info)
            self.complexity_ += 1

        self._recompute_leaves(X, y)
        return self

    @staticmethod
    def _make_node(info):
        node = ObliqueFIGSNode()
        if info["is_oblique"]:
            node.is_oblique = True; node.features = info["features"]; node.weights = info["weights"]
        else:
            node.feature = info["feature"]
        node.threshold = info["threshold"]
        node.left = ObliqueFIGSNode(value=info["left_value"], n_samples=info["n_left"])
        node.right = ObliqueFIGSNode(value=info["right_value"], n_samples=info["n_right"])
        node.n_samples = info["n_left"] + info["n_right"]
        return node

    @staticmethod
    def _split_leaf(leaf, info):
        if info["is_oblique"]:
            leaf.is_oblique = True; leaf.features = info["features"]; leaf.weights = info["weights"]
        else:
            leaf.is_oblique = False; leaf.feature = info["feature"]
        leaf.threshold = info["threshold"]
        leaf.left = ObliqueFIGSNode(value=info["left_value"], n_samples=info["n_left"])
        leaf.right = ObliqueFIGSNode(value=info["right_value"], n_samples=info["n_right"])

    def _recompute_leaves(self, X, y):
        n = len(y)
        for ti in range(len(self.trees_)):
            other = np.zeros(n)
            for tj, t in enumerate(self.trees_):
                if tj != ti:
                    other += _predict_tree(t, X)
            res = y - other
            leaves, masks = _get_leaves_and_masks(self.trees_[ti], X)
            for leaf, mask in zip(leaves, masks):
                idxs = np.where(mask)[0]
                leaf.value = float(np.mean(res[idxs])) if len(idxs) > 0 else 0.0

    def predict(self, X):
        if not self.trees_:
            return np.zeros(len(X))
        return sum(_predict_tree(t, X) for t in self.trees_)

    def predict_class(self, X):
        return (self.predict(X) > 0.5).astype(int)

    def predict_proba(self, X):
        raw = np.clip(self.predict(X), 0.0, 1.0)
        return np.column_stack([1 - raw, raw])


class SignedSpectralFIGS(BaseFIGSOblique):
    """Our method: draws feature subsets from SPONGE spectral modules."""
    def __init__(self, spectral_modules, **kwargs):
        super().__init__(**kwargs)
        self.spectral_modules = spectral_modules

    def _get_feature_subsets(self, d, rng):
        valid = [m for m in self.spectral_modules if len(m) >= 2]
        if not valid:
            return [sorted(rng.choice(d, size=min(self.beam_size, d), replace=False).tolist())
                    for _ in range(self.num_repetitions)]
        subsets = []
        for _ in range(self.num_repetitions):
            mod = valid[rng.integers(len(valid))]
            if len(mod) <= self.beam_size:
                subsets.append(sorted(mod))
            else:
                subsets.append(sorted(rng.choice(mod, size=self.beam_size, replace=False).tolist()))
        return subsets


class RandomObliqueFIGS(BaseFIGSOblique):
    """Baseline: random feature subsets of matched size."""
    def _get_feature_subsets(self, d, rng):
        return [sorted(rng.choice(d, size=min(self.beam_size, d), replace=False).tolist())
                for _ in range(self.num_repetitions)]


class AxisAlignedFIGSCustom(BaseFIGSOblique):
    """Fallback axis-aligned FIGS (no oblique subsets)."""
    def _get_feature_subsets(self, d, rng):
        return []

print("Oblique FIGS tree classes defined.")

## Metrics and Evaluation

Compute accuracy and interpretability metrics for each model: balanced accuracy, AUC-ROC, total splits, split arity, and path length.

In [ ]:
def compute_oblique_metrics(model, X_test, y_test, task_type, n_classes):
    """Compute accuracy + interpretability metrics."""
    metrics = {}
    if task_type == "classification":
        if n_classes and n_classes > 2:
            y_pred = model.predict(X_test)
        else:
            y_pred = model.predict_class(X_test) if hasattr(model, "predict_class") else model.predict(X_test)
        metrics["balanced_accuracy"] = float(balanced_accuracy_score(y_test, y_pred))
        if n_classes == 2:
            try:
                yp = model.predict_proba(X_test)[:, 1]
                metrics["auc_roc"] = float(roc_auc_score(y_test, yp)) if len(np.unique(yp)) > 1 else None
            except Exception:
                metrics["auc_roc"] = None
        else:
            metrics["auc_roc"] = None
    else:
        y_pred = model.predict(X_test)
        metrics["r2"] = float(r2_score(y_test, y_pred))

    # Interpretability
    total_splits, arities, depths = 0, [], []
    for root in (model.trees_ if hasattr(model, "trees_") else []):
        stack = [(root, 0)]
        while stack:
            nd, dep = stack.pop()
            if nd.is_leaf:
                depths.append(dep)
            else:
                total_splits += 1
                if nd.is_oblique and nd.weights is not None and len(nd.weights) > 0:
                    arities.append(max(int(np.sum(np.abs(nd.weights) > 1e-10)), 1))
                else:
                    arities.append(1)
                if nd.left: stack.append((nd.left, dep + 1))
                if nd.right: stack.append((nd.right, dep + 1))

    metrics["total_splits"] = total_splits
    metrics["n_trees"] = len(model.trees_) if hasattr(model, "trees_") else 0
    metrics["avg_split_arity"] = float(np.mean(arities)) if arities else 1.0
    metrics["max_split_arity"] = int(max(arities)) if arities else 1
    metrics["avg_path_length"] = float(np.mean(depths)) if depths else 0.0
    return metrics, y_pred

print("Metrics function defined.")

## Run Experiment

Execute the full pipeline for each dataset and fold:
1. Compute Co-Information matrix
2. Run SPONGE spectral clustering to find feature modules
3. Train all three FIGS variants (axis-aligned, random oblique, signed spectral)
4. Evaluate on held-out fold

In [ ]:
all_results = []
t_total = time.time()

for ds_name, ds in datasets.items():
    X_full, y_full = ds["X"], ds["y"]
    folds = ds["folds"]
    task_type = ds["task_type"]
    n_classes = ds.get("n_classes")
    d = X_full.shape[1]

    print(f"\n{'='*60}")
    print(f"Dataset: {ds_name}, n={len(y_full)}, d={d}, task={task_type}")
    print(f"{'='*60}")

    unique_folds = sorted(np.unique(folds).tolist())[:N_FOLDS]

    for fold_id in unique_folds:
        t_fold = time.time()
        test_mask = folds == fold_id
        train_mask = ~test_mask
        X_train, y_train = X_full[train_mask], y_full[train_mask]
        X_test, y_test = X_full[test_mask], y_full[test_mask]
        print(f"\n  Fold {fold_id}: train={len(y_train)}, test={len(y_test)}")

        # Phase 1: CoI matrix
        t0 = time.time()
        n_sub = min(len(y_train), COI_SUBSAMPLE_N)
        X_sub, y_sub = X_train[:n_sub], y_train[:n_sub]
        coi_matrix, individual_mi = compute_coi_matrix(X_sub, y_sub, task_type, k=COI_K_NEIGHBORS, n_jobs=N_JOBS)
        coi_time = time.time() - t0
        print(f"    CoI: {coi_time:.1f}s, shape={coi_matrix.shape}")

        # Phase 2: SPONGE clustering
        t1 = time.time()
        nz = coi_matrix[np.triu_indices(d, k=1)]
        nz = nz[np.abs(nz) > 1e-12]
        degenerate = True
        if len(nz) > 0:
            frac_pos = float(np.mean(nz > 0))
            degenerate = frac_pos > 0.95 or frac_pos < 0.05

        if degenerate:
            try:
                abs_coi = np.abs(coi_matrix)
                np.fill_diagonal(abs_coi, 0)
                k_fb = min(3, d - 1)
                sc = SpectralClustering(n_clusters=k_fb, affinity="precomputed", random_state=RANDOM_STATE)
                cluster_labels = sc.fit_predict(abs_coi)
            except Exception:
                cluster_labels = np.zeros(d, dtype=int)
                cluster_labels[d // 2:] = 1
            k_chosen = int(len(np.unique(cluster_labels)))
            silhouette_val = -1.0
        else:
            try:
                cluster_labels, _, k_chosen, silhouette_val = select_k_and_cluster(coi_matrix, k_max=SPONGE_K_MAX)
            except Exception as e:
                print(f"    SPONGE failed: {e}, using fallback")
                cluster_labels = np.zeros(d, dtype=int)
                cluster_labels[d // 2:] = 1
                k_chosen = 2
                silhouette_val = -1.0

        frustration = compute_frustration_index(coi_matrix)
        modules = extract_modules(cluster_labels, d)
        sponge_time = time.time() - t1
        module_sizes = [len(m) for m in modules]
        print(f"    SPONGE: k={k_chosen}, modules={module_sizes}, frustration={frustration:.4f}")

        # Phase 3: Scale data
        scaler = MinMaxScaler()
        X_tr_s = scaler.fit_transform(X_train)
        X_te_s = scaler.transform(X_test)
        beam_size = max(2, int(np.median(module_sizes)))

        # Phase 4: Train all methods
        for ms in MAX_SPLITS_GRID:
            base = {"dataset": ds_name, "fold": fold_id, "max_splits": ms,
                    "task_type": task_type, "n_classes": n_classes,
                    "frustration_index": float(frustration), "k_chosen": k_chosen,
                    "module_sizes": module_sizes}

            # M1: Axis-aligned FIGS
            t_m = time.time()
            try:
                aa = AxisAlignedFIGSCustom(max_splits=ms)
                aa.fit(X_tr_s, y_train)
                aa_met, _ = compute_oblique_metrics(aa, X_te_s, y_test, task_type, n_classes)
            except Exception as e:
                print(f"    AA FIGS failed: {e}")
                aa_met = {"balanced_accuracy": 0.5, "total_splits": 0, "n_trees": 0,
                          "avg_split_arity": 1.0, "max_split_arity": 1, "avg_path_length": 0.0}
            aa_met["fit_time_sec"] = float(time.time() - t_m)
            all_results.append({**base, "method": "axis_aligned_figs", **aa_met})

            # M2: Random oblique FIGS
            t_m = time.time()
            try:
                ro = RandomObliqueFIGS(max_splits=ms, beam_size=beam_size)
                ro.fit(X_tr_s, y_train)
                ro_met, _ = compute_oblique_metrics(ro, X_te_s, y_test, task_type, n_classes)
            except Exception as e:
                print(f"    RO FIGS failed: {e}")
                ro_met = {"balanced_accuracy": 0.5, "total_splits": 0, "n_trees": 0,
                          "avg_split_arity": 1.0, "max_split_arity": 1, "avg_path_length": 0.0}
            ro_met["fit_time_sec"] = float(time.time() - t_m)
            all_results.append({**base, "method": "random_oblique_figs", **ro_met})

            # M3: Signed spectral FIGS (ours)
            t_m = time.time()
            try:
                ss = SignedSpectralFIGS(spectral_modules=modules, max_splits=ms, beam_size=beam_size)
                ss.fit(X_tr_s, y_train)
                ss_met, _ = compute_oblique_metrics(ss, X_te_s, y_test, task_type, n_classes)
            except Exception as e:
                print(f"    SS FIGS failed: {e}")
                ss_met = {"balanced_accuracy": 0.5, "total_splits": 0, "n_trees": 0,
                          "avg_split_arity": 1.0, "max_split_arity": 1, "avg_path_length": 0.0}
            ss_met["fit_time_sec"] = float(time.time() - t_m)
            all_results.append({**base, "method": "signed_spectral_figs", **ss_met})

            # Quick log
            def _sc(m):
                return m.get("balanced_accuracy", m.get("r2", "?"))
            print(f"    ms={ms:2d}: AA={_sc(aa_met):.4f}  RO={_sc(ro_met):.4f}  SS={_sc(ss_met):.4f}")

        print(f"  Fold {fold_id} done in {time.time()-t_fold:.1f}s")

print(f"\nTotal experiment time: {time.time()-t_total:.1f}s")
print(f"Collected {len(all_results)} result rows.")

## Results Visualization

Summary table and bar chart comparing the three methods across all folds and complexity levels.

In [ ]:
import pandas as pd

# Build summary table
rows = []
for method in METHODS:
    method_rows = [r for r in all_results if r["method"] == method]
    if not method_rows:
        continue
    task = method_rows[0]["task_type"]
    if task == "classification":
        accs = [r["balanced_accuracy"] for r in method_rows]
        metric_name = "Balanced Accuracy"
    else:
        accs = [r["r2"] for r in method_rows]
        metric_name = "R2"
    splits = [r["total_splits"] for r in method_rows]
    arities = [r["avg_split_arity"] for r in method_rows]
    times = [r["fit_time_sec"] for r in method_rows]
    rows.append({
        "Method": method.replace("_", " ").title(),
        metric_name: f"{np.mean(accs):.4f} +/- {np.std(accs):.4f}",
        "Avg Splits": f"{np.mean(splits):.1f}",
        "Avg Arity": f"{np.mean(arities):.2f}",
        "Fit Time (s)": f"{np.mean(times):.3f}",
    })

df_summary = pd.DataFrame(rows)
print("\n=== RESULTS SUMMARY ===\n")
print(df_summary.to_string(index=False))

# Bar chart comparing methods
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Accuracy comparison
method_names = []
method_accs = []
method_stds = []
for method in METHODS:
    method_rows = [r for r in all_results if r["method"] == method]
    if not method_rows:
        continue
    task = method_rows[0]["task_type"]
    if task == "classification":
        accs = [r["balanced_accuracy"] for r in method_rows]
    else:
        accs = [r["r2"] for r in method_rows]
    method_names.append(method.replace("_", " ").replace("figs", "FIGS").title())
    method_accs.append(np.mean(accs))
    method_stds.append(np.std(accs))

colors = ['#4C72B0', '#55A868', '#C44E52']
bars = axes[0].bar(range(len(method_names)), method_accs, yerr=method_stds,
                   color=colors[:len(method_names)], capsize=5, alpha=0.85)
axes[0].set_xticks(range(len(method_names)))
axes[0].set_xticklabels(method_names, rotation=15, ha='right', fontsize=9)
axes[0].set_ylabel("Balanced Accuracy" if task == "classification" else "R2 Score")
axes[0].set_title("Method Comparison: Accuracy")
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Split arity comparison
method_arities = []
for method in METHODS:
    method_rows = [r for r in all_results if r["method"] == method]
    if not method_rows:
        continue
    method_arities.append(np.mean([r["avg_split_arity"] for r in method_rows]))

axes[1].bar(range(len(method_names)), method_arities,
            color=colors[:len(method_names)], alpha=0.85)
axes[1].set_xticks(range(len(method_names)))
axes[1].set_xticklabels(method_names, rotation=15, ha='right', fontsize=9)
axes[1].set_ylabel("Avg Split Arity")
axes[1].set_title("Method Comparison: Interpretability (Split Arity)")
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("results_comparison.png", dpi=100, bbox_inches='tight')
plt.show()
print("\nDone! Results saved.")